# Pothole detector - training on Colab

**Runtime -> Change runtime type -> T4 GPU.** Then Runtime -> Run all.

What happens: clone the repo, download the RDD2022 *India* subset from a Kaggle mirror
(~500 MB, no Kaggle login needed), keep only pothole boxes (class D40), fine-tune YOLOv8s,
and save `best.pt` to Google Drive.

Dataset: RDD2022 (Arya et al., 2022), India subset: 7,706 training images, 1,530 of them
contain potholes (3,187 boxes). The official `RDD2022_India.zip` link on the sekilab repo
returns 403 now, so we use the Kaggle mirror `musfequa/india-road-damage`, which has the
same folder layout (`India/train/images`, `India/train/annotations/xmls`).

In [ ]:
!pip -q install ultralytics kagglehub
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change runtime type to GPU")

In [ ]:
GITHUB_USER = 'HariOm297'
REPO = 'Pothole-Detection-System'
!rm -rf {REPO}
!git clone -q https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}

In [ ]:
# Download RDD2022 India (Kaggle mirror, ~500 MB) and convert to YOLO format (potholes only)
import kagglehub, pathlib
RDD_INDIA = pathlib.Path(kagglehub.dataset_download('musfequa/india-road-damage')) / 'India'
print("dataset at:", RDD_INDIA)
!python scripts/rdd_to_yolo.py --src {RDD_INDIA} --out data/yolo
# expected: positives=1530 background=153 train=1431 val=252

In [ ]:
# ~45 min on a T4. If Colab disconnects, re-run with --epochs 40.
!python scripts/train.py --data data/yolo/data.yaml --model yolov8s.pt --epochs 60 --imgsz 640 --batch 16 --name pothole_v1

In [ ]:
# Look at the training curves and a batch of validation predictions
from IPython.display import Image, display
import glob
runs = sorted(glob.glob('runs/**/pothole_v1*', recursive=True))
print("run dir:", runs[-1] if runs else "NOT FOUND - check the training log above")
run = runs[-1]
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{run}/{f}'
    if glob.glob(p):
        print(p); display(Image(p, width=900))


In [ ]:
# Save weights + curves to Drive so they survive the session
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pothole-models
!cp models/best.pt /content/drive/MyDrive/pothole-models/best.pt
!cp -r {run} /content/drive/MyDrive/pothole-models/
!ls -lh /content/drive/MyDrive/pothole-models

# Look at the training curves and a batch of validation predictions
from IPython.display import Image, display
import glob
runs = sorted(glob.glob('runs/**/pothole_v1*', recursive=True))
print("run dir:", runs[-1] if runs else "NOT FOUND - check the training log above")
run = runs[-1]
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{run}/{f}'
    if glob.glob(p):
        print(p); display(Image(p, width=900))
